##### ### The University of Melbourne, School of Computing and Information Systems
# COMP30027 Machine Learning, 2026 Semester 1

## Assignment 1: Income Classification with Naïve Bayes


**Student ID(s):**     `1610754`


This iPython notebook is a template which you will use for your Assignment 1 submission.

**NOTE: YOU SHOULD ADD YOUR RESULTS, GRAPHS, AND FIGURES FROM YOUR OBSERVATIONS IN THIS FILE TO YOUR REPORT (the PDF file).** Results, figures, etc. which appear in this file but are NOT included in your report will not be marked.

**Adding proper comments to your code is MANDATORY. **

## Setup and Preprocesing


In [6]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import ( 
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, roc_auc_score
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, CategoricalNB
from sklearn.preprocessing import OrdinalEncoder

In [7]:
DATA_DIR = "Assignment1_data"

CATEGORICAL_FEATURES = [
    "education", "marital-status", "native-country", "occupation",
    "race", "relationship", "sex", "workclass"
]
CONTINUOUS_FEATURES = [
    "age", "capital-gain", "capital-loss", "education-num",
    "fnlwgt", "hours-per-week"
]

ALL_FEATURES = CATEGORICAL_FEATURES + CONTINUOUS_FEATURES

INCOME = "income"


## Data Preprocessing


In [8]:
def load_and_clean(path: str, has_label: bool = True) -> tuple:
    """
    Load a CSV and clean it:
    - Missing values are coded as '?' or ' ?' — replace with NaN and drop those rows
    - Cast continuous columns to numeric
    - Strip whitespace from categorical columns
    Returns (X DataFrame, y Series) or (X DataFrame, None) for unlabelled data.
    """
    df = pd.read_csv(path)

    # Replace both '?' and ' ?' (with leading space) with NaN
    df = df.replace("?", np.nan)
    df = df.replace(" ?", np.nan)

    # Drop any row with a missing feature value
    df = df.dropna(subset=ALL_FEATURES)
    df = df.reset_index(drop=True)

    X = df[ALL_FEATURES].copy()

    # Ensure correct types
    for col in CONTINUOUS_FEATURES:
        X[col] = pd.to_numeric(X[col])
    for col in CATEGORICAL_FEATURES:
        X[col] = X[col].astype(str).str.strip()

    if has_label:
        y = df[INCOME].astype(str).str.strip()
        return X, y
    return X, None

In [9]:
# Load only the supervised training data needed for Task 1
X_train_full, y_train_full = load_and_clean(f'{DATA_DIR}/adult_supervised_train.csv')

print(f'Supervised train : {len(X_train_full)} rows')

# Carve out 15% as a validation set 
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.15, random_state=42, stratify=y_train_full,
)
print(f'\nTraining (after val split) : {len(X_train)} rows')
print(f'Validation: {len(X_val)} rows')

Supervised train : 15076 rows

Training (after val split) : 12814 rows
Validation: 2262 rows


In [12]:
# Check class imbalance 
print('Class distribution in trainin set:')
print(y_train_full.value_counts(normalize=True).map('{:.1%}'.format))


# Summary statistics for continuous features
X_train_full.describe()

Class distribution in trainin set:
income
<=50K    75.4%
>50K     24.6%
Name: proportion, dtype: str


,age,capital-gain,capital-loss,education-num,fnlwgt,hours-per-week
count,15076.000000,15076.000000,15076.000000,15076.000000,1.507600e+04,15076.000000
mean,38.744362,1005.850226,91.969422,10.112497,1.892799e+05,40.956487
std,13.290810,6970.042555,411.093674,2.561602,1.056658e+05,11.860822
min,17.000000,0.000000,0.000000,1.000000,1.349200e+04,1.000000
25%,28.000000,0.000000,0.000000,9.000000,1.168222e+05,40.000000
50%,37.000000,0.000000,0.000000,10.000000,1.779920e+05,40.000000
75%,48.000000,0.000000,0.000000,13.000000,2.386380e+05,45.000000
max,90.000000,99999.000000,3770.000000,16.000000,1.484705e+06,99.000000


## 1. Supervised model training


In [ ]:
class MixedNaiveBayes:
    """
    Mixed Naïve Bayes for continuous + categorical features.
    Uses GaussianNB for continuous features and CategoricalNB (Laplace-smoothed)
    for categorical features, combined in log-space.
    """

    def __init__(self, alpha: float = 1.0):
        self.alpha = alpha
        self.gnb = GaussianNB()
        self.cnb = CategoricalNB(alpha=alpha)
        # unknown_value=-1 flags unseen test categories for fallback handling
        self.encoder = OrdinalEncoder(
            handle_unknown="use_encoded_value", unknown_value=-1
        )

    def fit(self, X: pd.DataFrame, y: pd.Series):
        self.gnb.fit(X[CONTINUOUS_FEATURES].values, y)

        X_cat = self.encoder.fit_transform(X[CATEGORICAL_FEATURES])
        # Remap -1 (unseen) to 0 so CategoricalNB receives valid non-negative indices
        X_cat = np.where(X_cat < 0, 0, X_cat).astype(int)
        self.cnb.fit(X_cat, y)

        # classes_ is sorted alphabetically: index 0 = '<=50K', index 1 = '>50K'
        self.classes_ = list(self.gnb.classes_)

        # GaussianNB stores class_prior_ (not class_log_prior_); take log manually
        self.log_prior_ = {
            c: float(np.log(self.gnb.class_prior_[i]))
            for i, c in enumerate(self.classes_)
        }

        # gnb.theta_ / gnb.var_ have shape (n_classes, n_continuous_features)
        self.gauss_mean_ = {
            c: {feat: float(self.gnb.theta_[i, j]) for j, feat in enumerate(CONTINUOUS_FEATURES)}
            for i, c in enumerate(self.classes_)
        }
        self.gauss_var_ = {
            c: {feat: float(self.gnb.var_[i, j]) for j, feat in enumerate(CONTINUOUS_FEATURES)}
            for i, c in enumerate(self.classes_)
        }

        return self

    def _encode_categorical(self, X: pd.DataFrame) -> np.ndarray:
        X_cat = self.encoder.transform(X[CATEGORICAL_FEATURES])
        return np.where(X_cat < 0, 0, X_cat).astype(int)

    def predict_log_proba(self, X: pd.DataFrame) -> np.ndarray:
        """
        Both GaussianNB and CategoricalNB return log P(c) + log P(x|c),
        so subtract one copy of log_prior to avoid double-counting it.
        """
        lp_gnb = self.gnb.predict_log_proba(X[CONTINUOUS_FEATURES].values)
        lp_cnb = self.cnb.predict_log_proba(self._encode_categorical(X))
        return lp_gnb + lp_cnb - np.log(self.gnb.class_prior_)

    def predict_proba(self, X: pd.DataFrame) -> np.ndarray:
        lp = self.predict_log_proba(X)
        lp -= lp.max(axis=1, keepdims=True)  # shift for numerical stability
        p = np.exp(lp)
        return p / p.sum(axis=1, keepdims=True)

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        lp = self.predict_log_proba(X)
        return np.array([self.classes_[i] for i in lp.argmax(axis=1)])

    def confidence_ratio(self, X: pd.DataFrame) -> np.ndarray:
        """R = P(>50K|x) / P(<=50K|x); R>>1 confident >50K, R~1 near boundary."""
        lp = self.predict_log_proba(X)
        r = np.exp(lp[:, 1] - lp[:, 0])
        r[np.isinf(r)] = np.nan
        return r

    def top_predictive_categories(
        self, c1_idx: int = 1, c2_idx: int = 0, top_n: int = 5
    ):
        """
        Top category values for class c1 vs c2 ranked by R = P(v|c1)/P(v|c2).
        cnb.feature_log_prob_[fi] has shape (n_classes, n_categories).
        """
        rows = []
        for fi, feat in enumerate(CATEGORICAL_FEATURES):
            log_probs = self.cnb.feature_log_prob_[fi]
            for vi, v in enumerate(self.encoder.categories_[fi]):
                rows.append((feat, v, float(np.exp(log_probs[c1_idx, vi] - log_probs[c2_idx, vi]))))
        rows.sort(key=lambda x: -x[2])
        return rows[:top_n]


print("MixedNaiveBayes class defined.")

In [ ]:
sup_model = MixedNaiveBayes(alpha=1.0).fit(X_train, y_train)

print(f"Classes : {sup_model.classes_}")
print(f"Priors  : { {c: f'{np.exp(lp):.3f}' for c, lp in sup_model.log_prior_.items()} }")
print(f"Val acc : {accuracy_score(y_val, sup_model.predict(X_val)):.4f}")

In [11]:
# Build a pivot table showing mean and std for each continuous feature, split by class.
# Class separation can be assessed by comparing the mean difference to the within-class std.
rows = []
for feat in CONTINUOUS_FEATURES:
    for c in sup_model.classes_:
        mu  = sup_model.gauss_mean_[c][feat]
        std = np.sqrt(sup_model.gauss_var_[c][feat])
        rows.append({'Feature': feat, 'Class': c,
                     'Mean': f'{mu:.2f}', 'Std': f'{std:.2f}'})

df_params = pd.DataFrame(rows).pivot(
    index='Feature', columns='Class', values=['Mean', 'Std'])
df_params

NameError: name 'sup_model' is not defined

### 1.4 — Gaussian Distribution Plots

Visualising the learned Gaussian likelihoods helps assess which continuous features separate the two income classes most clearly.
**Include this figure in the report.**

## 2. Supervised model evaluation

## 3. Extending the model with semi-supervised training

## 4. Supervised model evaluation